In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# P48 fixed-target secondary continuation

The single authorized CPU validation on the saved P48 anchor and S/P/W/Q gradients
remains **NUMERICALLY_UNRESOLVED**: CLARABEL returned `optimal_inaccurate` in 11
iterations. Its stationarity residual is 2.6186873980371672e-6, also exceeding the
unchanged 1e-6 certificate criterion. No FP32 candidate was formed. No new GPU
execution is useful under this recorded decision.

The implementation eliminates secondary t and minimizes ||u||² subject to
h@u <= -0.3510973330917523, ||u|| <= 1 and ||ac+u|| <= 1. It uses the original
saved secondary_target, original normalization/two-pass span, CLARABEL 1e-10
tolerances and 200-iteration budget. It did not re-solve the primary problem or
change a tolerance, amplitude, solver or target. The new secondary primal/dual
and diagnostics are preserved; original primary primal/dual were never saved.

**Run all is a report-only continuation for this failed fixed CPU attempt.** It
actually reads anchor.pt, all four saved control_cotangent.pt files, the original
solution.json and ZERO/ANCHOR values from the existing Drive run. It rechecks the
single recorded secondary witness, preserves missing C and writes a separate
`secondary-direct-v1/run` directory beside the original `run`. It does not solve
again, load a model, decode, recompute any gradient or overwrite original records.
Repeated Run all reuses the stage's saved CPU decision and only rereads evidence.
The original 3-endpoint and 4-calibration denominators remain present.

Actual new-stage physical cost is **0**. If a future separately qualified single
C were available, a fresh physical stage would require 1 Transformer preparation,
1 VAE load, 2 ordinary DiT forwards, 2 native updates and 1 ordinary decode: 7
top-level calls, 60 original DiT blocks and 46 original decoder chunks, with zero
gradient calls or replay. The two loads are additional restart cost, not unused
old budget. That physical branch is not included/enabled in this unresolved
delivery; no hidden model retry exists.

Use the supplied ordinary editable source ZIP through Run all, or the standalone
CLI `python -m experiments.wan_state_clock.local_joint_p48_secondary_v1_run
--config <config.json> --decision <cpu_solution.json> --output <separate-stage>`.
No Git checkout, hashes, source manifests, exact version or GPU model is required.
File readability, numerical validity, fixed support and actual method quantities
are checked. Missing saved inputs are reported as saved-evidence errors; they do
not trigger regeneration. Return the entire new stage including failures.

This fixed-target formulation preserves the original secondary feasible u set.
Its observed numerical failure does not prove common descent infeasible. The
earlier primary positive local evidence remains; physical C and finite effects
remain unmeasured, scientific_pass remains false. Further method/numerical choices
are not adopted by this delivery.


In [ ]:
from pathlib import Path
import datetime, importlib, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_p48_secondary_v1_portable_source.zip'
FIXED_CONFIG = {'saved_run': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-P48-Joint-Control/resumable-p48-joint-control-v1/run'}
OUTPUT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-P48-Joint-Control/resumable-p48-joint-control-v1/secondary-direct-v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
ATTEMPT = OUTPUT / 'notebook_attempts' / STAMP
ATTEMPT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-P48-Secondary-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n')
    os.replace(tmp, path)
if (OUTPUT / 'fixed_config.json').exists():
    FIXED_CONFIG = json.loads((OUTPUT / 'fixed_config.json').read_text())
else:
    write_json(OUTPUT / 'fixed_config.json', FIXED_CONFIG)
fixed_slots = dict(status='SETUP_STARTED', objectives={n: dict(status='MISSING') for n in ('S','P','W','Q')}, endpoints={b: dict(status='MISSING', reason='not_run') for b in ('ZERO', 'ANCHOR', 'CANDIDATE')}, scientific_pass=False)
write_json(ATTEMPT / 'fixed_slots.json', fixed_slots)
def fail(stage, exc):
    write_json(ATTEMPT / 'failure.json', dict(stage=stage, reason=repr(exc), retry=False, traceback=traceback.format_exc()))
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    write_json(ATTEMPT / 'fixed_slots.json', fixed_slots)
def logged(command, stage, *, cwd=None):
    child = None
    try:
        with (ATTEMPT / 'execution.log').open('a') as log:
            print(stage, flush=True)
            child = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                     text=True, bufsize=1, start_new_session=True)
            for line in child.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            return child.wait()
    except BaseException as exc:
        fail(stage, exc)
        raise
    finally:
        if child is not None:
            if child.poll() is None:
                try: os.killpg(child.pid, signal.SIGTERM)
                except ProcessLookupError: pass
                try: child.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    try: os.killpg(child.pid, signal.SIGKILL)
                    except ProcessLookupError: pass
                    child.wait(timeout=10)
            if child.stdout is not None: child.stdout.close()
print('P48 unresolved secondary report task:', OUTPUT)

In [ ]:
import io, urllib.request, urllib.error, zipfile
SOURCE_CACHE = OUTPUT / 'source_package.zip'
try:
    downloaded = False
    try:
        with urllib.request.urlopen(SOURCE_URL, timeout=30) as response:
            package = response.read()
        downloaded = True
        source_location = SOURCE_URL
    except Exception as network_error:
        try:
            package = SOURCE_CACHE.read_bytes()
        except OSError as cache_error:
            raise RuntimeError('Source download unavailable: ' + repr(network_error)
                               + '; source cache unreadable: ' + repr(cache_error)) from cache_error
        source_location = str(SOURCE_CACHE)
        print('Network source unavailable; loading saved source ZIP:', repr(network_error))
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package)) as archive:
        for entry in ('runtime/wan/local_joint_p48_secondary_v1.py',
                      'experiments/wan_state_clock/local_joint_p48_secondary_v1_run.py'):
            try:
                archive.read(entry).decode('utf-8')
            except (KeyError, OSError, ValueError, UnicodeError) as entry_error:
                raise RuntimeError('Source entry unreadable: ' + entry + ': ' + repr(entry_error)) from entry_error
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise ValueError('unsafe source ZIP path: ' + member.filename)
            if member.is_dir(): target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    if downloaded:
        try:
            temporary = SOURCE_CACHE.with_suffix('.zip.tmp')
            temporary.write_bytes(package)
            os.replace(temporary, SOURCE_CACHE)
        except OSError as cache_error:
            print('Source loaded, but saving its offline cache failed:', repr(cache_error))
    for module_name in list(sys.modules):
        if module_name.split('.', 1)[0] in ('main', 'runtime', 'experiments'):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()
    sys.path.insert(0, str(WORKSPACE))
    print('Source location:', source_location, WORKSPACE)
except BaseException as exc:
    fail('SOURCE_LOAD', exc)
    raise

In [ ]:
# The fixed CPU outcome is unresolved. Only CPU tensor/array readers are
# needed; do not install diffusers/transformers or request CUDA/token.
try:
    probe = 'import torch, numpy'
    if logged([PYTHON, '-u', '-c', probe], 'CPU_DEPENDENCY_IMPORT'):
        if logged([PYTHON, '-m', 'pip', 'install', 'torch', 'numpy'], 'CPU_DEPENDENCY_INSTALL'):
            raise RuntimeError('CPU reader dependency installation failed')
        if logged([PYTHON, '-u', '-c', probe], 'CPU_DEPENDENCY_RECHECK'):
            raise RuntimeError('CPU reader import failed')
except BaseException as exc:
    fail('CPU_DEPENDENCIES', exc)
    raise
print('Recorded fixed-target CPU solve: optimal_inaccurate. No solver rerun or GPU work.')

In [ ]:
from runtime.wan.local_joint_p48_secondary_v1 import finalize_interrupted
rc = None
try:
    rc = logged([PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_p48_secondary_v1_run',
                 '--config', str(OUTPUT / 'fixed_config.json'), '--output', str(RUN_OUTPUT),
                 '--decision', str(WORKSPACE / 'notebooks/local_joint_p48_secondary_v1_cpu_solution.json')],
                'SAVED_SECONDARY_REPORT_ONLY', cwd=WORKSPACE)
finally:
    result = finalize_interrupted(RUN_OUTPUT, 'report reader interrupted; return code ' + str(rc))
    write_json(ATTEMPT / 'execution_receipt.json', dict(returncode=rc,
               result_status=None if result is None else result['status'], automatic_retry=False,
               model_calls=0, solver_calls=0))
if result is None: raise RuntimeError('No report; retain notebook attempt logs')
fixed_slots.update(status=result['status'], endpoints=result['endpoints'], objectives=result['objectives'])
write_json(ATTEMPT / 'fixed_slots.json', fixed_slots)

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text())
print('Fixed CPU outcome:', result['status'], '; scientific PASS: False')
print('New physical calls:', result['actual_new_stage_calls'], '; no model or solver rerun')
print('CPU validation used 0 primary solves and 1 fixed-target secondary solve; this notebook uses 0 solves.')
print('Original task retained at:', result['source_run'])
print('Original + new physical calls:', result.get('original_call_count'), '+ 0')
print('CANDIDATE:', result['endpoints']['CANDIDATE'])
print('Secondary receipt:', result.get('solution'))
print('Original planned 7-call candidate stage was NOT ENTERED:', result['physical_budget_if_qualified_candidate'])
print('Failures:', result['failures']); print('Resume:', result['resume_policy'])
print('Return report directory:', OUTPUT)